In [1]:
from pathlib import Path
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = 'iti-unesp-510015'
credential_file = Path.cwd() / 'iti-unesp-510015-1b5281550d01.json'

credentials = service_account.Credentials.from_service_account_file(
    credential_file
)
client = bigquery.Client(project=project_id, credentials=credentials)

query = """
SELECT 
  p.publication_number,
  p.filing_date,
  STRING_AGG(DISTINCT ipc_elem.code, '; ') AS ipc_codes,
  title_localized.text AS title,
  abstract_localized.text AS abstract
FROM 
  `patents-public-data.patents.publications` p,
  UNNEST(p.ipc) AS ipc_elem,
  UNNEST(p.title_localized) AS title_localized,
  UNNEST(p.abstract_localized) AS abstract_localized
WHERE 
  p.country_code = 'BR'
  AND title_localized.language = 'pt'
  AND abstract_localized.language = 'pt'
GROUP BY
  p.publication_number,
  p.filing_date,
  title_localized.text,
  abstract_localized.text
"""

# Executa a consulta e gera o DataFrame
df_automacao = client.query(query).to_dataframe()

print(f"Total de patentes de automação encontradas: {len(df_automacao)}")
print(df_automacao.head())

Total de patentes de automação encontradas: 781884
   publication_number  filing_date  \
0     BR-PI0500266-C1     20050131   
1  BR-112020007064-A2     20180620   
2  BR-112013006594-B1     20110920   
3  BR-112021026164-A2     20200625   
4  BR-112018070089-A2     20170331   

                                           ipc_codes  \
0  C08L23/06; B65D65/38; C08L23/04; C08K5/00; C08...   
1  D04H3/16; D06M17/00; A61F13/49; D04H3/007; B32...   
2  C07D239/88; A61P25/24; C07D409/14; C07D471/04;...   
3                    G07D11/13; G07F19/00; G07D11/18   
4                                           H04L1/00   

                                               title  \
0  processo para produção de filme polimérico tor...   
1  folha elástica para artigo absorvente e artigo...   
2  Composto ciclopropano antagonista de receptor ...   
3  Máquina de transação automatizada e cassete pa...   
4  métodos de operação de um dispositivo sem fio ...   

                                            ab

In [2]:
df_automacao

,publication_number,filing_date,ipc_codes,title,abstract
0,BR-PI0500266-C1,20050131,C08L23/06; B65D65/38; C08L23/04; C08K5/00; C08...,processo para produção de filme polimérico tor...,&#34;PROCESSO PARA PRODUçãO DE FILME POLIMéRIC...
1,BR-112020007064-A2,20180620,D04H3/16; D06M17/00; A61F13/49; D04H3/007; B32...,folha elástica para artigo absorvente e artigo...,A presente invenção refere-se a folha elástica...
2,BR-112013006594-B1,20110920,C07D239/88; A61P25/24; C07D409/14; C07D471/04;...,Composto ciclopropano antagonista de receptor ...,"COMPOSTO, COMPOSIÇÃO FARMACÊUTICA, MÉTODO DE T..."
3,BR-112021026164-A2,20200625,G07D11/13; G07F19/00; G07D11/18,Máquina de transação automatizada e cassete pa...,MÁQUINA DE TRANSAÇÃO AUTOMATIZADA E CASSETE PA...
4,BR-112018070089-A2,20170331,H04L1/00,métodos de operação de um dispositivo sem fio ...,são providos sistemas e métodos para relato da...
...,...,...,...,...,...
781879,BR-112015022632-B1,20140311,C22C21/00,"Composição de liga de magnésio, implante, e mé...",resumo patente de invenção &#34;liga de magnés...
781880,BR-112015028985-A2,20140521,B01D39/20,elemento de filtro cerâmico e método de fabric...,1/1 resumo elemento de filtro cerâmico e méto...
781881,BR-112016016939-A2,20140124,H04W72/12,"Método, aparelho e produto de programa de comp...","MÉTODO, APARELHO E PRODUTO DE PROGRAMA DE COMP..."
781882,BR-9814313-A,19981222,B68G9/00,Processo e aparelho para produzir um conjunto ...,&#34;PROCESSO E APARELHO PARA PRODUZIR UM CONJ...


In [3]:
df_automacao.info()

<class 'pandas.DataFrame'>
RangeIndex: 781884 entries, 0 to 781883
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   publication_number  781884 non-null  str  
 1   filing_date         781884 non-null  Int64
 2   ipc_codes           781884 non-null  str  
 3   title               781884 non-null  str  
 4   abstract            781884 non-null  str  
dtypes: Int64(1), str(4)
memory usage: 789.5 MB


In [4]:
df_automacao.to_excel('patentes_automacao_filtradas.xlsx', index=False)

In [5]:
df_automacao_pos2010 = df_automacao[df_automacao['filing_date'] >= 20100101]


In [6]:
df_automacao_pos2010

,publication_number,filing_date,ipc_codes,title,abstract
1,BR-112020007064-A2,20180620,D04H3/16; D06M17/00; A61F13/49; D04H3/007; B32...,folha elástica para artigo absorvente e artigo...,A presente invenção refere-se a folha elástica...
2,BR-112013006594-B1,20110920,C07D239/88; A61P25/24; C07D409/14; C07D471/04;...,Composto ciclopropano antagonista de receptor ...,"COMPOSTO, COMPOSIÇÃO FARMACÊUTICA, MÉTODO DE T..."
3,BR-112021026164-A2,20200625,G07D11/13; G07F19/00; G07D11/18,Máquina de transação automatizada e cassete pa...,MÁQUINA DE TRANSAÇÃO AUTOMATIZADA E CASSETE PA...
4,BR-112018070089-A2,20170331,H04L1/00,métodos de operação de um dispositivo sem fio ...,são providos sistemas e métodos para relato da...
5,BR-202021008981-U2,20210507,A22C7/00,Equipamento automático para montar espetinhos ...,EQUIPAMENTO AUTOMÁTICO PARA MONTAR ESPETINHOS ...
...,...,...,...,...,...
781876,BR-102017018829-A8,20170901,G06Q30/06,Sistema de busca de produtos e serviços com in...,SISTEMA DE BUSCA DE PRODUTOS E SERVIÇOS COM IN...
781878,BR-122013014461-B1,20100608,C25D5/10,revestimento de multicamadas resistente à corr...,COBERTURAS E REVESTIMENTOS NANOLAM INADAS ELET...
781879,BR-112015022632-B1,20140311,C22C21/00,"Composição de liga de magnésio, implante, e mé...",resumo patente de invenção &#34;liga de magnés...
781880,BR-112015028985-A2,20140521,B01D39/20,elemento de filtro cerâmico e método de fabric...,1/1 resumo elemento de filtro cerâmico e méto...


In [7]:
df_automacao_pos2010.info()

<class 'pandas.DataFrame'>
Index: 497466 entries, 1 to 781881
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   publication_number  497466 non-null  str  
 1   filing_date         497466 non-null  Int64
 2   ipc_codes           497466 non-null  str  
 3   title               497466 non-null  str  
 4   abstract            497466 non-null  str  
dtypes: Int64(1), str(4)
memory usage: 502.1 MB


In [8]:
df_automacao_limpo = df_automacao_pos2010.drop_duplicates(subset=['publication_number']).copy()
df_automacao_limpo['ipc_codes'] = (
    df_automacao_limpo['ipc_codes']
    .str.split(';')
    .str[0]
    .str.strip()
)

In [9]:
df_automacao_limpo.info()

<class 'pandas.DataFrame'>
Index: 384327 entries, 1 to 781881
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   publication_number  384327 non-null  str  
 1   filing_date         384327 non-null  Int64
 2   ipc_codes           384327 non-null  str  
 3   title               384327 non-null  str  
 4   abstract            384327 non-null  str  
dtypes: Int64(1), str(4)
memory usage: 387.0 MB


In [10]:
df_automacao_limpo.to_excel('df_automacao_limpo.xlsx', index=False)


In [11]:
df_automacao_limpo.to_csv('df_automacao_limpo.csv', index=False, sep=';')

In [12]:
ipc = df_automacao_limpo['ipc_codes']
ipc.head()

1    D04H3/16; D06M17/00; A61F13/49; D04H3/007; B32...
2    C07D239/88; A61P25/24; C07D409/14; C07D471/04;...
3                      G07D11/13; G07F19/00; G07D11/18
4                                             H04L1/00
5                                             A22C7/00
Name: ipc_codes, dtype: str

In [13]:
unicos_ipc = ipc.str.split('; ').explode().unique()
quatroipc = [ipc[:4] for ipc in unicos_ipc]

In [14]:
ipc4 = pd.DataFrame({'IPC': unicos_ipc, '4-digits IPC': quatroipc})


In [15]:
ipc4

,IPC,4-digits IPC
0,D04H3/16,D04H
1,D06M17/00,D06M
2,A61F13/49,A61F
3,D04H3/007,D04H
4,B32B5/26,B32B
...,...,...
53275,B28B21/18,B28B
53276,C07C69/017,C07C
53277,H02G15/12,H02G
53278,F24B1/192,F24B


In [16]:
ipc = ipc4['4-digits IPC'].unique()

In [17]:
ipc_4_todos = pd.DataFrame({'4-digitos IPC': ipc})

In [18]:
ipc_4_todos.to_excel('ipc_4_todos.xlsx', index=False)


In [19]:
ipc_4_todos.info()

<class 'pandas.DataFrame'>
RangeIndex: 651 entries, 0 to 650
Data columns (total 1 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   4-digitos IPC  651 non-null    str  
dtypes: str(1)
memory usage: 7.8 KB


In [20]:
codigos_andre = ['A99Z',
   'B64U',
   'B99Z',
   'E99Z',
   'F24S',
   'F24T',
   'F24V',
   'F99Z',
   'G06V',
   'G16B',
   'G16C',
   'G16H',
   'G16Y',
   'G16Z',
   'G99Z',
   'H10B',
   'H10D',
   'H10F',
   'H10H',
   'H10K',
   'H10N',
   'H10P',
   'H10W',
   'H99Z']

In [21]:
contagem_codigos_andre = (
    df_automacao_limpo['ipc_codes']
    .dropna()
    .str.split(';')
    .explode()
    .str.strip()
    .str[:4]
    .value_counts()
    .reindex(codigos_andre, fill_value=0)
    .rename_axis('codigo')
    .reset_index(name='ocorrencias')
    .sort_values(['ocorrencias', 'codigo'], ascending=[False, True])
    .reset_index(drop=True)
)

contagem_codigos_andre

,codigo,ocorrencias
0,G16H,1771
1,G06V,1256
2,H10N,1115
3,F24S,936
4,H10W,875
5,G16B,843
6,H10D,623
7,H10K,591
8,B64U,585
9,H10P,287


In [ ]:
'''
contagem_codigos_andre.to_excel(
    'contagem_codigos_andre.xlsx',
    index=False
)
'''

In [ ]:
import os

